In [ ]:
# Cell 1 — Imports and data loading
import pandas as pd
import numpy as np
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# Keep these CSV files in the same folder as this notebook, or change the paths.
engagement_file = "Engagement_history_improved.csv"
hcp_file = "HCP_master_updated.csv"

engagement_df = pd.read_csv(engagement_file, parse_dates=["engagement_date"])
hcp_df = pd.read_csv(hcp_file)

required_engagement = {"hcp_id", "channel", "engagement_date", "engagement_successful"}
missing = required_engagement - set(engagement_df.columns)
if missing:
    raise ValueError(f"Engagement file is missing columns: {sorted(missing)}")
if "hcp_id" not in hcp_df.columns:
    raise ValueError("HCP master file must contain 'hcp_id'.")

print(f"HCP master: {hcp_df.shape[0]:,} rows × {hcp_df.shape[1]} columns")
print(f"Engagement history: {engagement_df.shape[0]:,} rows × {engagement_df.shape[1]} columns")
display(engagement_df.head())

In [ ]:
# Cell 2 — Create one HCP × channel feature table
channels = ["rep_visit", "phone_call", "webinar", "email", "digital_ad"]
reference_date = engagement_df["engagement_date"].max()

engagement_summary = (
    engagement_df.groupby(["hcp_id", "channel"], as_index=False)
    .agg(
        interaction_count=("engagement_successful", "size"),
        success_rate=("engagement_successful", "mean"),
        last_engagement_date=("engagement_date", "max"),
    )
)
engagement_summary["recency_days"] = (
    reference_date - engagement_summary["last_engagement_date"]
).dt.days

channel_features = engagement_summary.pivot(
    index="hcp_id",
    columns="channel",
    values=["interaction_count", "success_rate", "recency_days"],
)
channel_features.columns = [f"{metric}_{channel}" for metric, channel in channel_features.columns]
channel_features = channel_features.reset_index()

for channel in channels:
    for metric in ["interaction_count", "success_rate"]:
        column = f"{metric}_{channel}"
        if column not in channel_features:
            channel_features[column] = 0.0
        channel_features[column] = pd.to_numeric(channel_features[column], errors="coerce").fillna(0.0)
    recency_column = f"recency_days_{channel}"
    if recency_column not in channel_features:
        channel_features[recency_column] = np.nan

print("Reference date:", reference_date.date())
display(channel_features.head())

In [ ]:
# Cell 3 — Create the three component scores for each channel
# These are the same inputs used by EIML before its first entropy calculation.
for channel in channels:
    count_col = f"interaction_count_{channel}"
    max_count = channel_features[count_col].max()
    channel_features[f"frequency_score_{channel}"] = (
        channel_features[count_col] / max_count if max_count > 0 else 0.0
    )

    recency = channel_features[f"recency_days_{channel}"].fillna(9999).astype(float)
    channel_features[f"recency_score_{channel}"] = np.where(
        channel_features[count_col] > 0,
        1 / (1 + recency / 30),
        0.0,
    )
    channel_features[f"success_score_{channel}"] = channel_features[f"success_rate_{channel}"].clip(0, 1)

display(channel_features[[
    "hcp_id", "frequency_score_email", "success_score_email", "recency_score_email"
]].head(10))

In [ ]:
# Cell 4 — Original arbitrary-weight model (retain this for comparison)
ARBITRARY_COMPONENT_WEIGHTS = {"frequency": 0.30, "success_rate": 0.50, "recency": 0.20}
ARBITRARY_CHANNEL_WEIGHTS = {
    "rep_visit": 0.30, "phone_call": 0.20, "webinar": 0.20,
    "email": 0.15, "digital_ad": 0.15,
}
assert np.isclose(sum(ARBITRARY_COMPONENT_WEIGHTS.values()), 1.0)
assert np.isclose(sum(ARBITRARY_CHANNEL_WEIGHTS.values()), 1.0)

for channel in channels:
    channel_features[f"arbitrary_channel_score_{channel}"] = (
        ARBITRARY_COMPONENT_WEIGHTS["frequency"] * channel_features[f"frequency_score_{channel}"]
        + ARBITRARY_COMPONENT_WEIGHTS["success_rate"] * channel_features[f"success_score_{channel}"]
        + ARBITRARY_COMPONENT_WEIGHTS["recency"] * channel_features[f"recency_score_{channel}"]
    )

channel_features["arbitrary_weighted_score"] = 100 * sum(
    channel_features[f"arbitrary_channel_score_{channel}"] * weight
    for channel, weight in ARBITRARY_CHANNEL_WEIGHTS.items()
)
channel_features["arbitrary_weighted_score"] = channel_features["arbitrary_weighted_score"].round(2)

display(pd.DataFrame({
    "component": ARBITRARY_COMPONENT_WEIGHTS.keys(),
    "arbitrary_weight": ARBITRARY_COMPONENT_WEIGHTS.values(),
}))
display(pd.DataFrame({"channel": ARBITRARY_CHANNEL_WEIGHTS.keys(), "arbitrary_weight": ARBITRARY_CHANNEL_WEIGHTS.values()}))

In [ ]:
# Cell 5 — First entropy calculation: component weights (frequency, success rate, recency)
def entropy_weights(data):
    """EIML entropy-weight method; returns entropy, diversification and normalized weights."""
    X = data.astype(float).fillna(0).clip(lower=0)
    epsilon = 1e-12
    column_sums = X.sum(axis=0)
    P = X.div(column_sums.replace(0, epsilon), axis=1)
    n = len(X)
    if n <= 1:
        raise ValueError("Entropy calculation requires more than one observation.")
    k = 1 / np.log(n)
    entropy = -k * (P * np.log(P + epsilon)).sum(axis=0)
    diversification = (1 - entropy).clip(lower=0)
    weights = diversification / diversification.sum()
    if weights.isna().any() or np.isclose(weights.sum(), 0):
        weights = pd.Series(1 / len(diversification), index=diversification.index)
    return pd.DataFrame({"entropy": entropy, "diversification": diversification, "weight": weights})

# Stack every HCP × channel observation, exactly like EIML's indicator_data construction.
indicator_data = pd.DataFrame({
    "frequency": np.concatenate([channel_features[f"frequency_score_{c}"].to_numpy() for c in channels]),
    "success_rate": np.concatenate([channel_features[f"success_score_{c}"].to_numpy() for c in channels]),
    "recency": np.concatenate([channel_features[f"recency_score_{c}"].to_numpy() for c in channels]),
})
indicator_weights = entropy_weights(indicator_data)
display(indicator_weights)
print("Component entropy-weight sum:", round(indicator_weights["weight"].sum(), 6))

In [ ]:
# Cell 6 — Apply component entropy weights, then calculate channel entropy weights (second time)
w_frequency = indicator_weights.loc["frequency", "weight"]
w_success = indicator_weights.loc["success_rate", "weight"]
w_recency = indicator_weights.loc["recency", "weight"]

for channel in channels:
    channel_features[f"entropy_channel_score_{channel}"] = (
        w_frequency * channel_features[f"frequency_score_{channel}"]
        + w_success * channel_features[f"success_score_{channel}"]
        + w_recency * channel_features[f"recency_score_{channel}"]
    )

entropy_channel_score_cols = [f"entropy_channel_score_{channel}" for channel in channels]
channel_score_data = channel_features[entropy_channel_score_cols].copy()
channel_score_data.columns = channels
channel_weights = entropy_weights(channel_score_data)

channel_features["entropy_weighted_score"] = 100 * sum(
    channel_features[f"entropy_channel_score_{channel}"] * channel_weights.loc[channel, "weight"]
    for channel in channels
)
channel_features["entropy_weighted_score"] = channel_features["entropy_weighted_score"].round(2)

display(channel_weights)
print("Channel entropy-weight sum:", round(channel_weights["weight"].sum(), 6))

In [ ]:
# Cell 7 — Compare models, validate and save the combined results
for score_name in ["arbitrary_weighted_score", "entropy_weighted_score"]:
    channel_features[score_name.replace("_score", "_rank")] = (
        channel_features[score_name].rank(method="min", ascending=False).astype(int)
    )
channel_features["score_difference_entropy_minus_arbitrary"] = (
    channel_features["entropy_weighted_score"] - channel_features["arbitrary_weighted_score"]
).round(2)

entropy_score_columns = [f"entropy_channel_score_{channel}" for channel in channels]
channel_features["recommended_channel"] = (
    channel_features[entropy_score_columns].idxmax(axis=1).str.replace("entropy_channel_score_", "", regex=False)
)
channel_features.loc[channel_features[entropy_score_columns].sum(axis=1).eq(0), "recommended_channel"] = "No engagement history"

final_dataset = hcp_df.merge(channel_features, on="hcp_id", how="left", validate="one_to_one")
if "opt_out_flag" in final_dataset.columns:
    final_dataset["engagement_eligible"] = ~final_dataset["opt_out_flag"].fillna(False)
    final_dataset.loc[~final_dataset["engagement_eligible"], "recommended_channel"] = "Do Not Contact"

comparison_columns = ["hcp_id", "arbitrary_weighted_score", "arbitrary_weighted_rank", "entropy_weighted_score", "entropy_weighted_rank", "score_difference_entropy_minus_arbitrary", "recommended_channel"]
display(final_dataset[comparison_columns].sort_values("entropy_weighted_score", ascending=False).head(20))
display(final_dataset[["arbitrary_weighted_score", "entropy_weighted_score", "score_difference_entropy_minus_arbitrary"]].describe())

output_file = "HCP_Arbitrary_and_Double_Entropy_Engagement_Scores.csv"
final_dataset.to_csv(output_file, index=False)
print(f"Saved combined dataset to: {output_file}")